# Expfit notes 4b: Special forms for multiple exponentials

When fitting more than one exponential, we want to restrict the values parameters can take.

1. With two growing exponentials, one will rapidly make the other unobservable, so beyond the single exponential case we want $c < 0$.
2. For the common use case of bi-exponential decay (or tri-exponential etc.), we want all $b > 0$ or all $b < 0$
3. Another common case is to have $m^+$ rapid exponentials in one direction, followed by $m^-$ slow exponentials in the other (e.g. $b_{i < m^+}>0,\ b_{i \geq m^+} < 0,\ c_i < 0$).

To achieve this without modifying the optimiser, we can use log transformed parameters.

## Restricting $c$

We set $\gamma=\log(-c) \rightarrow c=-e^\gamma$ and redefine the MSE as

\begin{align}
E_m = \frac{1}{n}\sum_{i=1}^n \left( y - a - \sum_{j=1}^m b_j e^{-e^{\gamma_j} x_i} \right)^2
\end{align}

The shorthand becomes
\begin{align}
e_j = e^{-e^{\gamma_j} x_i}
\end{align}
with partial derivative
\begin{align}
\frac{\partial}{\partial \gamma_j} e_j = -x_i e_j e^{\gamma_j}
\end{align}

Compared to $\frac{\partial}{\partial c_j} e^{c_j x_i} = x_i e^{c_j x_i}$, this means an extra multiplier $-e^\gamma$.

### Jacobian

\begin{align}
\frac{\partial}{\partial a} f &= 1
    & \rightarrow &&
    \frac{\partial E_m}{\partial a} &= \frac{2}{n} \sum_{i=1}^n f \\
\frac{\partial}{\partial b_s} f &= e_s
    & \rightarrow &&
    \frac{\partial E_m}{\partial b_s} &= \frac{2}{n} \sum_{i=1}^n f e_s \\
\frac{\partial}{\partial \gamma_s} f &= -b_s x e_s e^{\gamma_s}
    & \rightarrow &&
    \frac{\partial E_m}{\partial \gamma_s} &= -\frac{2 b_s e^{\gamma_s}}{n} \sum_{i=1}^n f x e_s
\end{align}

### Hessian

For $a$
\begin{align}
\frac{\partial^2 E_m}{\partial a^2} &= 2 \\
\frac{\partial^2 E_m}{\partial b_r\,\partial a} &= \frac{2}{n}\sum_{i=1}^n e_r \\
\frac{\partial^2 E_m}{\partial \gamma_r\,\partial a} &= -\frac{2 b_r e^{\gamma_s}}{n}\sum_{i=1}^n x e_s
\end{align}

For $b$
\begin{align}
\frac{\partial^2 E_m}{\partial b_r\,\partial b_s} &= \frac{2}{n}\sum_{i=1}^n e_r e_s \\
\end{align}

For $b$ and $\gamma$
\begin{align}
\frac{\partial^2 E_m}{\partial \gamma_s\,\partial b_s} &= -\frac{2}{n}\sum_{i=1}^n (f + b_s e_s) xe_se^{\gamma_s} \\
r \neq s, \frac{\partial^2 E_m}{\partial \gamma_r\,\partial b_s} &= -\frac{2 b_r e^{\gamma_r}}{n}\sum_{i=1}^n x e_r e_s
\end{align}

For $\gamma$
\begin{align}
r \neq s, \frac{\partial^2 E_m}{\partial \gamma_r \gamma_s} &= \frac{2 b_r b_s e^{\gamma_r} e^{\gamma_s}}{n}\sum_{i=1}^n x^2 e_r e_s \\
\frac{\partial^2 E_m}{\partial \gamma_s^2} &= \frac{2 b_s e^{\gamma_s}}{n}\sum_{i=1}^n 
\left( (f + b_se_s) x^2 e_s e^{\gamma_s} - f_sxe_s \right)
\end{align}

The final equation is more complicated than before, as it involves the derivative of $e_s e^{\gamma_s}$, which is a product of two functions of $\gamma_s$.

### Checking against finite differences

In [6]:
import numpy as np
import matplotlib.pyplot as plt

In [7]:
x = np.linspace(0, 1, 100)

a, b, ct = 5, 5, 0.3
y = a + b * np.exp(-np.exp(ct) * x)

a0, b0, ct0 = a + 1, b - 1, ct + 1

In [8]:
def mse(x, y, p):
    d = len(p)
    assert (d - 1) % 2 == 0 and d > 1
    m = (d - 1) // 2
    p = np.asarray(p)    
    bs = p[1::2].reshape((m, 1))
    cs = -np.exp(p[2::2].reshape((m, 1)))
    return np.sum((p[0] - y + np.sum(bs * np.exp(cs * x), axis=0))**2) / len(x)


def mse_jac_fd(x, y, p, dp):
    e = mse(x, y, p)
    jac = np.zeros(len(p))
    p = np.array(p, dtype=float)
    for i in range(len(p)):
        q = np.copy(p)
        q[i] += dp
        jac[i] = (mse(x, y, q) - e) / dp
    return e, jac
    
    
def mse_jac_hes_fd(x, y, p, dp=1e-5):
    d = len(p)
    m = (d - 1) // 2
    mse, jac = mse_jac_fd(x, y, p, dp)
    
    hes = np.zeros((d, d))
    p = np.array(p, dtype=float)
    for i in range(len(p)):
        q = np.copy(p)
        q[i] += dp
        hes[i] = (mse_jac_fd(x, y, q, dp)[1] - jac) / dp   
    return mse, jac, hes


In [12]:
def mse_jac_hes(x, y, p):
    d = len(p)
    assert (d - 1) % 2 == 0 and d > 1
    m = (d - 1) // 2

    n = len(x)
    ninv2 = 2 / n
    
    # Unpack
    p = np.asarray(p)
    a = p[0]
    b = p[1::2].reshape((m, 1))       # (m, 1)
    ct = p[2::2].reshape((m, 1))      # (m, 1)
    
    # MSE
    ect = np.exp(ct)
    e = np.exp(-ect * x)            # (m, n)
    be = b * e                      # (m, n)
    f = a - y + np.sum(be, axis=0)  # (n, )
    mse = np.sum(f**2) / n

    # Jacobian
    ex = e * x * ect
    jac = np.zeros(d)
    jac[0] = ninv2 * np.sum(f)
    jac[1::2] = ninv2 * np.sum(f * e, axis=1)
    jac[2::2] = -ninv2 * np.sum(f * ex, axis=1) * b.T

    # Hessian
    hes = np.zeros((d, d))
    # aa, ab, ac
    hes[0, 0] = 2
    hes[0, 1::2] = hes[1::2, 0] = ninv2 * np.sum(e, axis=1)
    hes[0, 2::2] = hes[2::2, 0] = -ninv2 * np.sum(ex, axis=1) * b.T
    for i in range(m):
        # bi^2, ci^2, and bi*ci
        hes[1 + 2 * i, 1 + 2 * i] = ninv2 * np.sum(e[i]**2)       
        hes[2 + 2 * i, 2 + 2 * i] = ninv2 * b[i, 0] * np.sum(
            ((f + be[i]) * x * ect[i] - f) * ex[i])
        hes[1 + 2 * i, 2 + 2 * i] = hes[2 + 2 * i, 1 + 2 * i] = \
            -ninv2 * np.sum((f + be[i]) * ex[i])
        for j in range(i + 1, m):
            # bi*bj, ci*cj, bi*cj, bj*ci
            hes[1 + 2 * i, 1 + 2 * j] = hes[1 + 2 * j, 1 + 2 * i] = \
                ninv2 * np.sum(e[i] * e[j])
            hes[2 + 2 * i, 2 + 2 * j] = hes[2 + 2 * j, 2 + 2 * i] = \
                ninv2 * np.sum(ex[i] * ex[j]) * b[i, 0] * b[j, 0]
            hes[1 + 2 * i, 2 + 2 * j] = hes[2 + 2 * j, 1 + 2 * i] = \
                -ninv2 * np.sum(ex[j] * e[i]) * b[j, 0]
            hes[2 + 2 * i, 1 + 2 * j] = hes[1 + 2 * j, 2 + 2 * i] = \
                -ninv2 * np.sum(ex[i] * e[j]) * b[i, 0]

    return mse, jac, hes


m1, j1, h1 = mse_jac_hes(x, y, (a0, b0, ct0, 3, -7, 1.1, 2.2))
m2, j2, h2 = mse_jac_hes_fd(x, y, (a0, b0, ct0, 3, -7, 1.1, 2.2))
print(m1)
print(m2)
print()
with np.printoptions(linewidth=120):
    print(j1)
    print(j2)
    print()
    print(np.max(np.abs(h1 - h2)))

6.180662833781528
6.180662833781528

[ 4.89883194  1.43602515 -4.41990583  4.89663664 -0.00658384  0.70968099 -0.60486045]
[ 4.89884194  1.43602655 -4.41988182  4.89664663 -0.00658387  0.70968159 -0.60485768]

4.6534207494342006e-05


6.180662833781528
6.180662833781528

[ 4.89883194  1.43602515 -4.41990583  4.89663664 -0.00658384  0.70968099 -0.60486045]
[ 4.89884194  1.43602655 -4.41988182  4.89664663 -0.00658387  0.70968159 -0.60485768]

[[ True  True  True  True  True  True  True]
 [ True  True  True  True  True  True  True]
 [ True  True  True  True  True  True  True]
 [ True  True  True  True  True  True  True]
 [ True  True  True  True  True  True  True]
 [ True  True  True  True  True  True  True]
 [ True  True  True  True  True  True  True]]


## Restricting $c$ and $b$

Next, we add similar restrictions on $b$, by writing $b_j = z_j e^{\gamma_j}$, where $z_j \in {-1, 1}$ is a constant determining the sign of $b$.

This gives us the MSE
\begin{align}
E_m = \frac{1}{n}\sum_{i=1}^n \left( y - a - \sum_{j=1}^m z_j e^{\beta_j} e^{-e^{\gamma_j} x_i} \right)^2
\end{align}

### Jacobian

\begin{align}
\frac{\partial}{\partial a} f &= 1
    & \rightarrow &&
    \frac{\partial E_m}{\partial a} &= \frac{2}{n} \sum_{i=1}^n f \\
\frac{\partial}{\partial \beta_s} f &= z_s e^{\beta_s} e_s
    & \rightarrow &&
    \frac{\partial E_m}{\partial \beta_s} &= \frac{2z_s}{n} \sum_{i=1}^n f e^{\beta_s} e_s \\
\frac{\partial}{\partial \gamma_s} f &= -z_s e^{\beta_s} x e_s e^{\gamma_s}
    & \rightarrow &&
    \frac{\partial E_m}{\partial \gamma_s} &= -\frac{2z_se^{\beta_s}e^{\gamma_s}}{n} \sum_{i=1}^n f x e_s
\end{align}

\begin{align}
\frac{\partial E_m}{\partial a} &= \frac{2}{n}\sum_{i=1}^n f
\end{align}

\begin{align}
\frac{\partial E_m}{\partial \beta_s} &= \frac{2}{n}\sum_{i=1}^n f z_s e^{\beta_s} e_s
\end{align}

\begin{align}
\frac{\partial E_m}{\partial \gamma_s} &= -\frac{2 z_s e^{\beta_s}}{n}\sum_{i=1}^n f x e_s e^{\gamma_s}
\end{align}

### Hessian

For a
\begin{align}
\frac{\partial^2 E_m}{\partial a^2} &= 2 \\
\frac{\partial^2 E_m}{\partial \beta_r\,\partial a} &= \frac{2}{n}\sum^n z_r e^{\beta_r} e_r \\
\frac{\partial^2 E_m}{\partial \gamma_r\,\partial a} &= -\frac{2b_r}{n}\sum^n x e_s e^{q_s}
\end{align}
